# Dataset 5 — Wind & Solar Energy Production (Kaggle)

**Etapa B — Python / Pandas**

Comparação da ocorrência de períodos de alta produção solar e alta produção eólica, cada uma avaliada em relação ao seu próprio valor máximo (70% do máximo individual como limiar).

## 0. Carregamento dos dados

O arquivo `amostra.csv` é a amostra de 20% exportada na Etapa A (Orange). O dataset está em **formato longo**: cada linha representa um registro de produção de **uma única fonte** (coluna `Source`, com valores `Wind` ou `Solar`), e o valor correspondente fica na coluna `Production`. Não existem, portanto, duas colunas separadas de geração no arquivo original — elas serão construídas no passo 1.

In [1]:
import pandas as pd

df = pd.read_csv('amostra.csv')
df.head()

,Date,Start_Hour,End_Hour,Source,Day_of_Year,Day_Name,Month_Name,Season,Production
0,12/18/2022,21,22,Wind,352,Sunday,December,Winter,11400
1,1/29/2024,14,15,Wind,29,Monday,January,Winter,7917
2,8/1/2024,15,16,Solar,214,Thursday,August,Summer,8835
3,11/10/2020,1,2,Wind,315,Tuesday,November,Fall,989
4,12/19/2022,1,2,Wind,353,Monday,December,Winter,12526


## 1. Renomeando as variáveis principais

Como o dataset está no formato longo (uma coluna `Source` + uma coluna `Production`), a forma equivalente a "renomear as variáveis principais para `Geracao_Solar` e `Geracao_Eolica`" é **separar os registros por fonte** e nomear cada série resultante de acordo.

In [2]:
Geracao_Solar = df.loc[df['Source'] == 'Solar', 'Production'].rename('Geracao_Solar')
Geracao_Eolica = df.loc[df['Source'] == 'Wind', 'Production'].rename('Geracao_Eolica')

print("Registros de Geracao_Solar:", len(Geracao_Solar))
print("Registros de Geracao_Eolica:", len(Geracao_Eolica))

Registros de Geracao_Solar: 1846
Registros de Geracao_Eolica: 8527


## 2. Valor máximo registrado para cada fonte

In [3]:
max_solar = Geracao_Solar.max()
max_eolica = Geracao_Eolica.max()

print(f"Máximo Geracao_Solar:  {max_solar}")
print(f"Máximo Geracao_Eolica: {max_eolica}")

Máximo Geracao_Solar:  16316
Máximo Geracao_Eolica: 22929


## 3. Cálculo de 70% do máximo de cada fonte

In [4]:
limite_solar = 0.70 * max_solar
limite_eolica = 0.70 * max_eolica

print(f"Limite de alta geração Solar (70% do máximo):  {limite_solar:.2f}")
print(f"Limite de alta geração Eólica (70% do máximo): {limite_eolica:.2f}")

Limite de alta geração Solar (70% do máximo):  11421.20
Limite de alta geração Eólica (70% do máximo): 16050.30


## 4. DataFrames de alta geração (por fonte)

In [5]:
alta_solar = df[(df['Source'] == 'Solar') & (df['Production'] >= limite_solar)]
alta_eolica = df[(df['Source'] == 'Wind') & (df['Production'] >= limite_eolica)]

print("Amostra de registros de alta geração Solar:")
display(alta_solar.head())

print("\nAmostra de registros de alta geração Eólica:")
display(alta_eolica.head())

Amostra de registros de alta geração Solar:


,Date,Start_Hour,End_Hour,Source,Day_of_Year,Day_Name,Month_Name,Season,Production
77,7/8/2025,13,14,Solar,189,Tuesday,July,Summer,13125
178,3/19/2025,13,14,Solar,78,Wednesday,March,Spring,13786
183,8/17/2025,12,13,Solar,229,Sunday,August,Summer,11455
187,7/16/2025,12,13,Solar,197,Wednesday,July,Summer,16160
207,3/31/2025,12,13,Solar,90,Monday,March,Spring,12441



Amostra de registros de alta geração Eólica:


,Date,Start_Hour,End_Hour,Source,Day_of_Year,Day_Name,Month_Name,Season,Production
20,1/3/2024,0,1,Wind,3,Wednesday,January,Winter,17272
32,3/11/2021,13,14,Wind,70,Thursday,March,Spring,16144
103,1/29/2025,0,1,Wind,29,Wednesday,January,Winter,18199
151,12/8/2024,6,7,Wind,343,Sunday,December,Winter,16946
192,11/19/2024,2,3,Wind,324,Tuesday,November,Fall,16568


## 5. Contagem e percentuais de alta geração por fonte

In [6]:
n_solar_total = len(Geracao_Solar)
n_eolica_total = len(Geracao_Eolica)

n_alta_solar = len(alta_solar)
n_alta_eolica = len(alta_eolica)

pct_solar = n_alta_solar / n_solar_total * 100
pct_eolica = n_alta_eolica / n_eolica_total * 100

resumo = pd.DataFrame({
    'Fonte': ['Solar', 'Eólica'],
    'Registros totais': [n_solar_total, n_eolica_total],
    'Registros de alta geração (>=70% do máximo)': [n_alta_solar, n_alta_eolica],
    'Percentual (%)': [round(pct_solar, 2), round(pct_eolica, 2)]
})
resumo

,Fonte,Registros totais,Registros de alta geração (>=70% do máximo),Percentual (%)
0,Solar,1846,45,2.44
1,Eólica,8527,245,2.87


## 6. Comparação entre as fontes

In [7]:
if pct_solar > pct_eolica:
    fonte_mais_frequente = 'Solar'
elif pct_eolica > pct_solar:
    fonte_mais_frequente = 'Eólica'
else:
    fonte_mais_frequente = 'Empate'

print(f"Percentual de alta geração — Solar: {pct_solar:.2f}%")
print(f"Percentual de alta geração — Eólica: {pct_eolica:.2f}%")
print(f"\nFonte com maior frequência relativa de alta geração (acima de 70% do seu próprio máximo): {fonte_mais_frequente}")

Percentual de alta geração — Solar: 2.44%
Percentual de alta geração — Eólica: 2.87%

Fonte com maior frequência relativa de alta geração (acima de 70% do seu próprio máximo): Eólica


**Resultado interpretativo:**

Na amostra analisada, a produção **eólica** apresentou uma frequência relativa de alta geração (2,87%) ligeiramente **maior** do que a produção **solar** (2,44%), quando cada uma é comparada apenas com seu próprio valor máximo histórico. A diferença é pequena, o que sugere que, proporcionalmente, as duas fontes atingem seus picos individuais com frequência semelhante nesta amostra — a eólica alcança seu teto de geração (22.929) um pouco mais frequentemente, em termos relativos, do que a solar alcança o dela (16.316).

## 7. Por que não usar o mesmo valor numérico de potência como limite para as duas fontes?

Utilizar diretamente o mesmo valor absoluto (por exemplo, "considerar alta geração acima de 10.000") como limiar para solar e eólica seria inadequado porque as duas variáveis possuem **escalas e distribuições diferentes**:

- O máximo observado de geração eólica (22.929) é bem maior que o máximo solar (16.316) nesta amostra — as fontes não operam na mesma faixa de valores.
- Um limite fixo, definido olhando apenas para uma das escalas, distorceria a comparação: um valor que representa um pico extremo para a fonte solar poderia representar um valor mediano ou frequente para a fonte eólica (e vice-versa).
- Isso faria a análise refletir apenas a diferença de escala entre as fontes, e não a real frequência de "picos de produção" de cada uma.

Por isso, o critério correto é relativo: comparar cada fonte com **seu próprio máximo** (70% do máximo individual), como feito nas etapas 2 a 6. Esse ajuste normaliza a comparação e permite avaliar de forma justa qual fonte atinge seus próprios picos de geração com maior frequência relativa — que é exatamente o objetivo do operador do portfólio.